# Module 01: NumPy for Machine Learning
## Notebook 03: Vectorization and Broadcasting Rules

Vectorization and broadcasting are the twin engines that power modern numerical computing and machine learning. By eliminating explicit Python `for` loops, your code executes directly at compiled C speeds and scales effortlessly to millions of data points.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the architectural mechanics of **vectorization** and benchmark its speedup over loops.
2. Utilize universal functions (**ufuncs**) for mathematical transformations and gradient clipping.
3. Master the **3 General Broadcasting Rules** and anticipate resulting array shapes.
4. Perform conditional filtering and boolean masking using `np.where()` and `np.select()`.
5. Extract top predictions and ranking indices using `np.argmax()` and `np.argsort()`.

In [1]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3


### 1. The Vectorization Advantage

In standard Python, executing a loop requires the Python interpreter to inspect the type of every single object at each iteration, check method tables, and handle dynamic dispatch.

In NumPy:
- Operations are executed over contiguous memory blocks using compiled C/Fortran routines.
- SIMD (**Single Instruction, Multiple Data**) processor instructions execute arithmetic operations across multiple array elements simultaneously.

In [2]:
# Simulating Euclidean Distance between two feature vectors of 2,000,000 values
N = 2_000_000
u = np.random.rand(N)
v = np.random.rand(N)

# 1. Loop-based computation
start = time.time()
dist_loop = 0.0
for i in range(N):
    diff = u[i] - v[i]
    dist_loop += diff * diff
dist_loop = dist_loop ** 0.5
loop_time = time.time() - start

# 2. Vectorized computation
start = time.time()
dist_vec = np.sqrt(np.sum((u - v) ** 2))
vec_time = time.time() - start

print(f"Loop Distance:       {dist_loop:.4f} (took {loop_time:.4f}s)")
print(f"Vectorized Distance: {dist_vec:.4f} (took {vec_time:.4f}s)")
print(f"Vectorization Speedup: {loop_time / vec_time:.1f}x faster!")

Loop Distance:       577.4083 (took 8.1875s)
Vectorized Distance: 577.4083 (took 0.1457s)
Vectorization Speedup: 56.2x faster!


---
### 2. Universal Functions (ufuncs)

A universal function (**ufunc**) operates on `ndarray`s element-by-element.
Essential ufuncs in Machine Learning:
- Exponential & Logarithm: `np.exp()` (softmax / sigmoid), `np.log()`, `np.log1p()` (log loss, entropy).
- Power & Root: `np.sqrt()`, `np.square()`, `np.power()` (MSE, RMSE).
- Clipping & Clamping: `np.clip()` (preventing numerical explosion and exploding gradients).

In [3]:
logits = np.array([-10.0, -2.5, 0.0, 2.5, 10.0, 100.0])

# Sigmoid activation function: 1 / (1 + exp(-z))
# Using np.clip to prevent overflow encountered in exp
clipped_logits = np.clip(logits, -50.0, 50.0)
sigmoid_probs = 1.0 / (1.0 + np.exp(-clipped_logits))

print("Original Logits:       ", logits)
print("Sigmoid Probabilities: ", np.round(sigmoid_probs, 4))

# Safe log loss evaluation with np.clip
predictions = np.array([0.999, 0.001, 0.85, 0.40])
safe_preds = np.clip(predictions, 1e-15, 1 - 1e-15)
log_loss = -np.log(safe_preds)
print("\nCross-Entropy Losses:   ", np.round(log_loss, 4))

Original Logits:        [-10.   -2.5   0.    2.5  10.  100. ]
Sigmoid Probabilities:  [0.     0.0759 0.5    0.9241 1.     1.    ]

Cross-Entropy Losses:    [1.0000e-03 6.9078e+00 1.6250e-01 9.1630e-01]


---
### 3. The 3 General Broadcasting Rules

Broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations.

#### The Rules:
1. **Right-Alignment**: Compare the dimensions of both arrays starting from the **trailing (rightmost)** dimension and work backward.
2. **Compatibility**: Two dimensions are compatible if:
   - They are equal, OR
   - One of them is 1.
3. **Expansion**: Dimensions of size 1 are virtually stretched (without copying memory) to match the larger dimension. If dimensions do not match and neither is 1, NumPy raises a `ValueError: operands could not be broadcast together`.

```text
Example 1: (3, 4) + (4,)
   Array A:   3  x  4
   Array B:         4  (padded to 1 x 4)
   Result:    3  x  4  (Valid!)

Example 2: (4, 1) * (1, 5)
   Array A:   4  x  1
   Array B:   1  x  5
   Result:    4  x  5  (Outer product!)

Example 3: (3, 4) + (3,)  --> INCOMPATIBLE!
   Array A:   3  x  4
   Array B:         3  (4 != 3 and neither is 1 -> FAILS)
   Fix: Reshape B to (3, 1)!
```

In [4]:
# Example 1: Subtracting feature means from dataset X (centering)
# Dataset: 4 samples, 3 features
X = np.array([
    [10.0, 20.0, 30.0],
    [12.0, 24.0, 32.0],
    [14.0, 22.0, 28.0],
    [16.0, 26.0, 34.0]
])

feature_means = np.mean(X, axis=0) # Shape: (3,)

print(f"X shape:             {X.shape}")
print(f"feature_means shape: {feature_means.shape}")

# Broadcasting (4, 3) - (3,) -> feature_means is broadcast across all 4 rows
X_centered = X - feature_means
print("\nMean-centered feature matrix:\n", X_centered)

X shape:             (4, 3)
feature_means shape: (3,)

Mean-centered feature matrix:
 [[-3. -3. -1.]
 [-1.  1.  1.]
 [ 1. -1. -3.]
 [ 3.  3.  3.]]


In [5]:
# Example 2: Outer product using (M, 1) and (1, N)
# Compute pairwise multiplication grid
u = np.array([1, 2, 3])[:, np.newaxis]  # Shape: (3, 1)
v = np.array([10, 20, 30, 40])[np.newaxis, :]  # Shape: (1, 4)

grid = u * v  # Broadcasts to (3, 4)
print("u shape:", u.shape)
print("v shape:", v.shape)
print("Broadcasted product grid (3, 4):\n", grid)

u shape: (3, 1)
v shape: (1, 4)
Broadcasted product grid (3, 4):
 [[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


---
### 4. Boolean Masking and Conditional Filtering

Boolean masking allows you to select, count, and modify elements based on complex conditional logic without writing loops.
- Relational operators: `>`, `<`, `>=`, `<=`, `==`, `!=`
- Logical bitwise operators: `&` (AND), `|` (OR), `~` (NOT). *Parentheses around each sub-clause are mandatory!*
- `np.where(condition, value_if_true, value_if_false)`: Vectorized ternary operator.
- `np.select(conditions_list, choice_list, default)`: Multi-way branching.

In [6]:
scores = np.array([45, 88, 72, 95, 30, 60, 82, 91])

# 1. Creating a boolean mask
passing_mask = scores >= 60
print("Passing mask:       ", passing_mask)

# 2. Filtering with boolean indexing
passing_scores = scores[passing_mask]
print("Passing scores only:", passing_scores)

# 3. Compound condition: High performers (scores between 80 and 95 inclusive)
elite_mask = (scores >= 80) & (scores <= 95)
print("Elite scores (80-95):", scores[elite_mask])

# 4. np.where: Labeling Pass vs Fail
labels = np.where(scores >= 60, "PASS", "FAIL")
print("Pass/Fail labels:   ", labels)

# 5. Multi-way grading using np.select
conditions = [
    scores >= 90,
    (scores >= 75) & (scores < 90),
    (scores >= 60) & (scores < 75)
]
grades = ["A", "B", "C"]
assigned_grades = np.select(conditions, grades, default="F")
print("Letter grades:      ", assigned_grades)

Passing mask:        [False  True  True  True False  True  True  True]
Passing scores only: [88 72 95 60 82 91]
Elite scores (80-95): [88 95 82 91]
Pass/Fail labels:    ['FAIL' 'PASS' 'PASS' 'PASS' 'FAIL' 'PASS' 'PASS' 'PASS']
Letter grades:       ['F' 'B' 'C' 'A' 'F' 'C' 'B' 'A']


---
### 5. Searching and Sorting Operations

In machine learning, we constantly need to:
- Find the class with highest probability (`np.argmax()`).
- Rank predictions or find the $K$ nearest neighbors (`np.argsort()`).
- Check if all data meets quality criteria (`np.all()`, `np.any()`).

In [7]:
# Simulated Softmax Output for 4 samples across 3 classes (Class 0: Cat, Class 1: Dog, Class 2: Bird)
probs = np.array([
    [0.10, 0.70, 0.20],  # Sample 0 -> Dog
    [0.85, 0.05, 0.10],  # Sample 1 -> Cat
    [0.15, 0.25, 0.60],  # Sample 2 -> Bird
    [0.30, 0.40, 0.30]   # Sample 3 -> Dog
])

# Argmax along axis 1 (across columns) gives the winning class index for each sample
predicted_classes = np.argmax(probs, axis=1)
confidence_scores = np.max(probs, axis=1)

class_names = np.array(["Cat", "Dog", "Bird"])
print("Predicted class indices: ", predicted_classes)
print("Predicted class names:   ", class_names[predicted_classes])
print("Confidence scores:       ", confidence_scores)

# Argsort for ranking feature importances (ascending order)
feature_importances = np.array([0.12, 0.45, 0.03, 0.28, 0.12])
sorted_indices = np.argsort(feature_importances)[::-1] # descending
print("\nFeature ranking from most to least important:", sorted_indices)

Predicted class indices:  [1 0 2 1]
Predicted class names:    ['Dog' 'Cat' 'Bird' 'Dog']
Confidence scores:        [0.7  0.85 0.6  0.4 ]

Feature ranking from most to least important: [1 3 4 0 2]


### Summary & Next Steps
In this notebook, you mastered:
- The performance superiority of vectorization over interpreter loops.
- Universal functions (`ufuncs`) and safe numerical transformations.
- The 3 rules of broadcasting and how to debug dimension mismatches.
- Conditional indexing, `np.where()`, and `np.select()`.
- Model prediction decoding with `np.argmax()` and `np.argsort()`.

**Next Notebook:** `04_math_stats_and_linear_algebra.ipynb` — Deep dive into matrix multiplications, linear systems, eigenvalues (PCA foundation), and modern random number generation.